In [1]:
import os
import re
import numpy as np
import pandas as pd
import anndata as ad

from sklearn.decomposition import PCA
from sklearn.metrics import calinski_harabasz_score



In [ ]:

# -------------------------
# Settings
# -------------------------
input_dir = "/Users/apple/Desktop/KB/data/LarryData/Larry_simulation/Larry_simulation_4scvi/final_final"
output_csv = os.path.join(input_dir, "PCA_CH_scores_rawcount.csv")

n_components = 32  # match your embedding dimension


# -------------------------
# Helper: extract beta
# -------------------------
def parse_beta_from_filename(fname):
    m = re.search(r"Larry_Simulation_(\d+)_final_final\.h5ad$", fname)
    if m is None:
        return None
    return float(m.group(1)) / 10.0


# -------------------------
# Main
# -------------------------
files = sorted(f for f in os.listdir(input_dir) if f.endswith(".h5ad"))

results = []

for fname in files:
    path = os.path.join(input_dir, fname)
    print(f"\nProcessing: {fname}")

    adata = ad.read_h5ad(path)

    # Use clone_id directly
    labels = adata.obs["clone_id"].to_numpy()

    # Convert to dense if needed
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else np.asarray(adata.X)

    print(f"  data shape: {X.shape}")
    print(f"  #lineages: {len(np.unique(labels))}")

    # PCA
    pca = PCA(n_components=n_components, random_state=42)
    X_pca = pca.fit_transform(X)

    # CH score
    ch_score = calinski_harabasz_score(X_pca, labels)

    beta = parse_beta_from_filename(fname)

    results.append({
        "file": fname,
        "beta": beta,
        "CH_score": ch_score,
        "n_cells": X.shape[0],
        "n_lineages": len(np.unique(labels))
    })

    print(f"  PCA({n_components}) CH score: {ch_score:.4f}")

# -------------------------
# Save + print
# -------------------------
df = pd.DataFrame(results).sort_values("beta")
df.to_csv(output_csv, index=False)

print("\n====================")
print("Final PCA results")
print("====================")
print(df.to_string(index=False))

print(f"\nSaved to: {output_csv}")


Processing: Larry_Simulation_01_final_final.h5ad
  data shape: (41093, 2000)
  #lineages: 1221
  PCA(32) CH score: 169.9344

Processing: Larry_Simulation_03_final_final.h5ad
  data shape: (41093, 2000)
  #lineages: 970
  PCA(32) CH score: 220.7297

Processing: Larry_Simulation_05_final_final.h5ad
  data shape: (41093, 2000)
  #lineages: 797
  PCA(32) CH score: 263.3312

Processing: Larry_Simulation_07_final_final.h5ad
  data shape: (41093, 2000)
  #lineages: 597
  PCA(32) CH score: 339.1884

Processing: Larry_Simulation_09_final_final.h5ad
  data shape: (41093, 2000)
  #lineages: 385
  PCA(32) CH score: 502.2869

Final PCA results
                                file  beta   CH_score  n_cells  n_lineages
Larry_Simulation_01_final_final.h5ad   0.1 169.934431    41093        1221
Larry_Simulation_03_final_final.h5ad   0.3 220.729706    41093         970
Larry_Simulation_05_final_final.h5ad   0.5 263.331236    41093         797
Larry_Simulation_07_final_final.h5ad   0.7 339.188436    410

In [4]:
arr = np.array([169.934, 220.729, 263.331, 339.188, 502.286])
arrlog = np.log(arr)
arrlog

array([5.13541013, 5.3969357 , 5.5734118 , 5.82655453, 6.21916968])